#### Extract data from wri

In [9]:
import requests
import psycopg2
import csv
import io
import tempfile
import os

# URL of the Global Power Plant Database CSV file
data_url = "https://github.com/wri/global-power-plant-database/raw/master/output_database/global_power_plant_database.csv"

# List of European countries
EUROPEAN_COUNTRIES = {
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina", "Bulgaria",
    "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", "Latvia", "Liechtenstein", "Lithuania",
    "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia",
    "Norway", "Poland", "Portugal", "Romania", "San Marino", "Serbia", "Slovakia", "Slovenia",
    "Spain", "Sweden", "Switzerland", "Ukraine", "United Kingdom", "Vatican City"
}

# PostgreSQL database connection details
DB_NAME = "test"
DB_USER = "test"
DB_PASS = "test"
DB_HOST = "localhost"
TABLE_NAME = "european_power_plants"

def check_database_connection():
    """Check if we can connect to the database"""
    try:
        conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASS, host=DB_HOST)
        conn.close()
        return True
    except Exception as e:
        print(f"Database connection error: {e}")
        return False

def create_table(cursor, headers):
    """Create the database table based on CSV headers"""
    cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME};")
    columns_with_types = ", ".join([f'"{header}" TEXT' for header in headers])
    cursor.execute(f"CREATE TABLE {TABLE_NAME} ({columns_with_types});")
    print(f"Table {TABLE_NAME} created successfully")

def process_with_copy():
    """Download CSV, filter European data, and use COPY for bulk insert"""
    conn = None
    temp_file = None
    
    # First, check database connection
    if not check_database_connection():
        print("Cannot proceed without database connection")
        return
    
    try:
        # Download the CSV file with explicit streaming
        print("Downloading data...")
        response = requests.get(data_url, stream=True)
        response.raise_for_status()
        
        # Read all content and decode
        content = response.content.decode('utf-8')
        
        # Verify we have content
        if not content or len(content) < 100:  # Basic sanity check
            print("Error: Downloaded content appears to be empty or too small")
            print(f"Content preview: {content[:100]}")
            return
            
        # Parse the CSV to get headers and filter European countries
        print("Filtering data for European countries...")
        csv_data = io.StringIO(content)
        reader = csv.reader(csv_data)
        
        # Get headers
        try:
            headers = next(reader)
            print(f"Headers found: {headers}")
        except StopIteration:
            print("Error: Could not read headers from CSV")
            return
        
        # Find the country column index
        if "country" not in headers:
            print("Error: No 'country' column found in headers")
            return
            
        country_index = headers.index("country_long")
        
        # Create a temporary file for the filtered European data
        temp_file = tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.csv')
        writer = csv.writer(temp_file)
        writer.writerow(headers)  # Write headers
        
        # Filter rows for European countries only
        european_rows = 0
        total_rows = 0
        
        for row in reader:
            total_rows += 1
            if len(row) == len(headers) and row[country_index].strip() in EUROPEAN_COUNTRIES:
                writer.writerow(row)
                european_rows += 1
                
        temp_file.close()
        
        if european_rows == 0:
            print(f"Warning: No European power plants found among {total_rows} total rows")
            return
            
        print(f"Filtered {european_rows} European power plant entries from {total_rows} total")
        
        # Connect to PostgreSQL
        conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASS, host=DB_HOST)
        cursor = conn.cursor()
        
        # Create the table
        create_table(cursor, headers)
        
        # Use COPY command for bulk insert
        print("Performing bulk insert with COPY...")
        with open(temp_file.name, 'r') as f:
            # Skip the header row as we already created the table
            next(f)
            cursor.copy_expert(
                f"COPY {TABLE_NAME} FROM STDIN WITH CSV",
                f
            )
        
        # Verify data was inserted
        cursor.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
        count = cursor.fetchone()[0]
        
        conn.commit()
        print(f"Successfully imported {count} rows into the database")
        
        if count == 0:
            print("Warning: No rows were inserted into the database")
        elif count != european_rows - 1:  # -1 because we skip the header in COPY
            print(f"Warning: Expected {european_rows-1} rows but inserted {count}")
        
    except Exception as e:
        print(f"Error: {e}")
        if conn:
            conn.rollback()
    finally:
        if conn:
            cursor.close()
            conn.close()
            print("Database connection closed")
        
        # Clean up the temporary file
        if temp_file and os.path.exists(temp_file.name):
            os.unlink(temp_file.name)
            print("Temporary file deleted")

if __name__ == "__main__":
    print("Starting bulk import of European power plant data...")
    process_with_copy()
    print("Import process completed")

Starting bulk import of European power plant data...
Filtering data for European countries...
Headers found: ['country', 'country_long', 'name', 'gppd_idnr', 'capacity_mw', 'latitude', 'longitude', 'primary_fuel', 'other_fuel1', 'other_fuel2', 'other_fuel3', 'commissioning_year', 'owner', 'source', 'url', 'geolocation_source', 'wepp_id', 'year_of_capacity_data', 'generation_gwh_2013', 'generation_gwh_2014', 'generation_gwh_2015', 'generation_gwh_2016', 'generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019', 'generation_data_source', 'estimated_generation_gwh_2013', 'estimated_generation_gwh_2014', 'estimated_generation_gwh_2015', 'estimated_generation_gwh_2016', 'estimated_generation_gwh_2017', 'estimated_generation_note_2013', 'estimated_generation_note_2014', 'estimated_generation_note_2015', 'estimated_generation_note_2016', 'estimated_generation_note_2017']
Filtered 10207 European power plant entries from 34936 total
Table european_power_plants created successfully
Per

#### Extract data from eurostat
URLs:
1) Population by age groups (done)
https://ec.europa.eu/eurostat/databrowser/view/tps00010/default/table?lang=en&category=t_demo.t_demo_ind
2) Population on January 1st
https://ec.europa.eu/eurostat/databrowser/view/tps00001/default/table?lang=en&category=t_demo.t_demo_pop
3) Electricity prices
https://ec.europa.eu/eurostat/databrowser/view/nrg_pc_204/default/table?lang=en&category=nrg.nrg_price.nrg_pc


In [10]:
import requests
import psycopg2
import csv
import io
import tempfile
import os

# URL of the Eurostat CSV data
data_url = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/tps00010/1.0?compress=false&format=csvdata&formatVersion=2.0&lang=en&labels=name"

# PostgreSQL database connection details
DB_NAME = "test"
DB_USER = "test"
DB_PASS = "test"
DB_HOST = "localhost"
TABLE_NAME = "eurostat_data"

def check_database_connection():
    """Check if we can connect to the database"""
    try:
        conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASS, host=DB_HOST)
        conn.close()
        return True
    except Exception as e:
        print(f"Database connection error: {e}")
        return False

def create_table(cursor, headers):
    """Create the database table based on CSV headers"""
    cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME};")
    columns_with_types = ", ".join([f'"{header}" TEXT' for header in headers])
    cursor.execute(f"CREATE TABLE {TABLE_NAME} ({columns_with_types});")
    print(f"Table {TABLE_NAME} created successfully")

def process_eurostat_data():
    """Download Eurostat CSV data and use COPY for bulk insert"""
    conn = None
    temp_file = None
    
    # First, check database connection
    if not check_database_connection():
        print("Cannot proceed without database connection")
        return
    
    try:
        # Download the CSV file
        print("Downloading Eurostat data...")
        response = requests.get(data_url, stream=True)
        response.raise_for_status()
        
        # Read all content and decode
        content = response.content.decode('utf-8')
        
        # Verify we have content
        if not content or len(content) < 100:
            print("Error: Downloaded content appears to be empty or too small")
            print(f"Content preview: {content[:100]}")
            return
            
        # Parse the CSV to get headers
        print("Processing Eurostat data...")
        csv_data = io.StringIO(content)
        reader = csv.reader(csv_data)
        
        # Get headers
        try:
            headers = next(reader)
            print(f"Headers found: {len(headers)} columns")
            print(f"First few headers: {headers[:5]}...")
        except StopIteration:
            print("Error: Could not read headers from CSV")
            return
        
        # Create a temporary file for the data
        temp_file = tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.csv')
        writer = csv.writer(temp_file)
        writer.writerow(headers)  # Write headers
        
        # Copy all rows to the temporary file
        total_rows = 0
        for row in reader:
            if len(row) == len(headers):  # Ensure data integrity
                writer.writerow(row)
                total_rows += 1
                
        temp_file.close()
        
        if total_rows == 0:
            print("Warning: No data rows found in the CSV")
            return
            
        print(f"Processed {total_rows} data rows")
        
        # Connect to PostgreSQL
        conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASS, host=DB_HOST)
        cursor = conn.cursor()
        
        # Create the table
        create_table(cursor, headers)
        
        # Use COPY command for bulk insert
        print("Performing bulk insert with COPY...")
        with open(temp_file.name, 'r') as f:
            # Skip the header row as we already created the table
            next(f)
            cursor.copy_expert(
                f"COPY {TABLE_NAME} FROM STDIN WITH CSV",
                f
            )
        
        # Verify data was inserted
        cursor.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
        count = cursor.fetchone()[0]
        
        conn.commit()
        print(f"Successfully imported {count} rows into the database")
        
        if count == 0:
            print("Warning: No rows were inserted into the database")
        elif count != total_rows:
            print(f"Warning: Expected {total_rows} rows but inserted {count}")
        
    except Exception as e:
        print(f"Error: {e}")
        if conn:
            conn.rollback()
    finally:
        if conn:
            cursor.close()
            conn.close()
            print("Database connection closed")
        
        # Clean up the temporary file
        if temp_file and os.path.exists(temp_file.name):
            os.unlink(temp_file.name)
            print("Temporary file deleted")

if __name__ == "__main__":
    print("Starting import of Eurostat data...")
    process_eurostat_data()
    print("Import process completed")

Starting import of Eurostat data...
Processing Eurostat data...
Headers found: 17 columns
First few headers: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency']...
Processed 3250 data rows
Table eurostat_data created successfully
Performing bulk insert with COPY...
Successfully imported 3250 rows into the database
Database connection closed
Temporary file deleted
Import process completed
